In [2]:
import pandas as pd
from transformers import pipeline

# 1. Load test Dataset

In [4]:
test_data = pd.read_csv("../data/processed/test_data.csv")

In [62]:
# let's make a copy of it
test_results = test_data.copy()
texts = test_results["text"].tolist()
true_labels = test_results["sentiment"].tolist()
original_indexes = test_results["original_index"].tolist()

# 2. Load pre-trained Model

In [6]:
# Also let's get the predicted_labels and confidence score from the model
# let's load the FinBERT using the pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [34]:
model_pred = pipe(texts, batch_size=32)
pred_labels = [obj['label'] for obj in model_pred]
pred_score = [obj['score'] for obj in model_pred]

# 3. Analysis dataframe 
This dataframe contains columns of text, true_labels and from the model we have pred_labels for model predicted labels and pred_score for the model confidence in the prediction

In [64]:
analysis_obj = {
    "original_index": original_indexes,
    "text" : texts,
    "true_label": true_labels,
    "predicted_label": pred_labels,
    "pred_score": pred_score
}
analysis_df = pd.DataFrame(data=analysis_obj)

In [65]:
analysis_df

,original_index,text,true_label,predicted_label,pred_score
0,2636,"The mill will have capacity to produce 500,000 tonnes of pulp per year .",neutral,neutral,0.841666
1,547,"HELSINKI ( AFX ) - Nokian Tyres reported a fourth quarter pretax profit of 61.5 mln eur , up from 48.6 mln on the back of strong sales .",positive,positive,0.953589
2,4685,The company confirmed its estimate for lower revenue for the whole 2009 than the year-ago EUR93 .9 m as given in the interim report on 5 August 2009 .,negative,negative,0.972156
3,3071,"Prior to the transaction , whose financial terms have not been disclosed , Alma Media owned 40 % of Kotikokki net .",neutral,neutral,0.952597
4,2988,"L+Ænnen Tehtaat 's Food Division was reorganised into two strategic business units , Apetit Frozen Foods and Jams , and Apetit Fish .",neutral,neutral,0.938348
...,...,...,...,...,...
718,4472,"The previously concluded adaptation measures , concerning other staff , were adequate for the time being , and the planning operations continue as before at the plant , the company said .",positive,positive,0.921316
719,4807,"Operating result for the 12-month period decreased from the profit of EUR0 .4 m while turnover decreased from EUR5 .6 m , as compared to 2004 .",negative,negative,0.974548
720,3528,Combined net sales in 2006 were $ 27 million and EBITDA was $ 13.7 million .,neutral,neutral,0.944923
721,4553,"Operating profit for the six-month period decreased from EUR111 .9 m , while sales increased from EUR1 ,275 m , as compared to the corresponding period in 2006 .",neutral,negative,0.973749


# 4. Error analysis dataframe

In [66]:
# let's check how many of the predictions predict wrong and why
errors_df = analysis_df[
    analysis_df["true_label"] != analysis_df["predicted_label"]
]
errors_df

,original_index,text,true_label,predicted_label,pred_score
17,2166,"Pre-tax loss totaled EUR 0.3 mn , compared to a loss of EUR 2.2 mn in the first quarter of 2005 .",positive,negative,0.593626
24,1536,"In the next few years , the ICT sector 's share of electricity consumption will be raised by the increase in the popularity of smartphones .",neutral,positive,0.845783
33,3778,The company said that it has agreed to a EUR160m unsecured credit line from lenders .,neutral,positive,0.929986
42,222,Its total annual revenue comes up to about 160 mln zloty ( $ 56.9 mln-42 .3 mln euro ) .,neutral,positive,0.800894
86,3169,"The company has decided to stop the operations of Ruukki Construction Division in Latvia and Lithuania , and concentrate the production and logistics in Parnu , Estonia in 2009 .",neutral,negative,0.853950
...,...,...,...,...,...
692,4575,"ASPOCOMP GROUP OYJ STOCK EXCHANGE RELEASE December 15 , 2006 at 4:50 PM According to the disclosure received today by Aspocomp Group Oyj , the share of Henrik Nyberg in Aspocomp Group Oyj 's share capital and votes has decreased below 5 percent .",neutral,negative,0.922802
695,2700,` Nordic infrastructure construction is one of our strategic growth areas .,neutral,positive,0.639643
703,1988,"As a result of the cancellation , the maximum increase of Citycon 's share capital on the basis of the convertible bonds decreased from EUR 23,383,927.80 to EUR 22,901,784.75 .",neutral,negative,0.542261
707,468,`` The sale of the oxygen measurement business strengthens our goal to focus on our chosen market segments .,neutral,positive,0.936913


In [67]:
# let's check the distribution
errors_df.groupby(["true_label", "predicted_label"]).size()

true_label  predicted_label
negative    neutral             2
neutral     negative           26
            positive           40
positive    negative            2
            neutral            16
dtype: int64

In [68]:
pd.set_option("display.max_colwidth", None)
# let's check the wrong predictions with highest confidence scores, if any
errors_df.sort_values(
    "pred_score",
    ascending=False
).head(20)

,original_index,text,true_label,predicted_label,pred_score
721,4553,"Operating profit for the six-month period decreased from EUR111 .9 m , while sales increased from EUR1 ,275 m , as compared to the corresponding period in 2006 .",neutral,negative,0.973749
389,59,"In Sweden , Gallerix accumulated SEK denominated sales were down 1 % and EUR denominated sales were up 11 % .",neutral,negative,0.972846
613,2401,"At 10.58 am , Outokumpu declined 2.74 pct to 24.87 eur , while the OMX Helsinki 25 was 0.55 pct higher at 2,825.14 and the OMX Helsinki added 0.64 pct to 9,386.89 .",neutral,negative,0.951658
342,4464,The board further said the company omitted to tender for a substantial part of the works and as such they had rightfully been found non-responsive by the evaluation team .,neutral,negative,0.942037
137,2451,"firm 28 October 2009 - Finnish lifting equipment maker Konecranes Oyj HEL : KCR1V said today it acquired US Machine Tool Solutions Unlimited in Cincinnati , Ohio , for an undisclosed sum .",positive,neutral,0.938728
707,468,`` The sale of the oxygen measurement business strengthens our goal to focus on our chosen market segments .,neutral,positive,0.936913
453,3186,"The company will also shut one paper machine in Finland and one in Austria , as well as two label paper machines in Finland for up to 10 months , Helsinki-based UPM said yesterday .",neutral,negative,0.934186
649,949,Finnish mobile operator DNA will function as a subcontractor to Maingate and will be responsible for telecommunications connections .,positive,neutral,0.930086
33,3778,The company said that it has agreed to a EUR160m unsecured credit line from lenders .,neutral,positive,0.929986
372,1585,s already good position in the technical building services market in Ostrobothnia .,neutral,positive,0.928239


In [79]:
# Now from the analysis, neutral -> positive prediction is the highest, let's inspect them
neutral_positive_errors = errors_df[
    (errors_df["true_label"] == "neutral") & (errors_df["predicted_label"] == "positive")
]
neutral_positive_errors

,original_index,text,true_label,predicted_label,pred_score
24,1536,"In the next few years , the ICT sector 's share of electricity consumption will be raised by the increase in the popularity of smartphones .",neutral,positive,0.845783
33,3778,The company said that it has agreed to a EUR160m unsecured credit line from lenders .,neutral,positive,0.929986
42,222,Its total annual revenue comes up to about 160 mln zloty ( $ 56.9 mln-42 .3 mln euro ) .,neutral,positive,0.800894
102,3272,The new location is n't the only change Wellmont has in store for its air transport service .,neutral,positive,0.602828
159,3310,The prices of stainless steel also rose in Europe .,neutral,positive,0.909573
173,1480,"3G data subscribers are also helping to maintain these growth levels , since data-only subscriptions push for more multiple SIM ownership .",neutral,positive,0.888020
178,3945,"Vacon aims to establish its presence in the solar energy business in various parts of the world towards the end of 2010 , said Olli Teva , marketing director renewable energy applications .",neutral,positive,0.823138
184,1746,"Mr. Atul Chopra , Chief Operating Officer & President , Tecnotree elaborated '' I am eager to see the continued success of the former Tecnomen , bringing Lifetree product to MENA , under the Tecnotree umbrella .",neutral,positive,0.849241
242,2976,Joint procurement will be later extended to the factories in the Baltic countries .,neutral,positive,0.696376
252,1854,"So far the company has awarded more than $ 350,000 worth of tools and materials .",neutral,positive,0.676479


In [74]:
# among these 40, let's check the top confident score predictions
neutral_positive_errors.sort_values(
    "pred_score",
    ascending=False
).head(20)

,original_index,text,true_label,predicted_label,pred_score
707,468,`` The sale of the oxygen measurement business strengthens our goal to focus on our chosen market segments .,neutral,positive,0.936913
33,3778,The company said that it has agreed to a EUR160m unsecured credit line from lenders .,neutral,positive,0.929986
372,1585,s already good position in the technical building services market in Ostrobothnia .,neutral,positive,0.928239
159,3310,The prices of stainless steel also rose in Europe .,neutral,positive,0.909573
574,510,The company plans to spend the proceeds from the rights offering for strengthening its balance sheet .,neutral,positive,0.900021
340,3571,"Following the issue , the number of shares in the Swedish company will grow by 9 % .",neutral,positive,0.891750
173,1480,"3G data subscribers are also helping to maintain these growth levels , since data-only subscriptions push for more multiple SIM ownership .",neutral,positive,0.888020
315,4493,"Amer , which bought Salomon from adidas in October , said the job cuts are aimed at boosting competitiveness .",neutral,positive,0.886473
671,3662,"M-real Corporation Stock Exchange Announcement 29 September 2006 at 4.15 p.m. Kyro Corporation and M-real Corporation , a Metsaliitto Group subsidiary , have agreed on an arrangement which gives M-real option to purchase the Kyroskoski natural gas powerplant from Kyro .",neutral,positive,0.882493
675,399,Pohjola could increase its stake to 45 % in 2013 .,neutral,positive,0.876672


In [78]:
# Now in the same way let's check  neutral -> negative predictions, let's inspect them
neutral_negative_errors  = errors_df[
    (errors_df["true_label"] == "neutral") & ((errors_df["predicted_label"] == "negative"))
]
neutral_negative_errors.sort_values(
    "pred_score",
    ascending=False
).head(13)


,original_index,text,true_label,predicted_label,pred_score
721,4553,"Operating profit for the six-month period decreased from EUR111 .9 m , while sales increased from EUR1 ,275 m , as compared to the corresponding period in 2006 .",neutral,negative,0.973749
389,59,"In Sweden , Gallerix accumulated SEK denominated sales were down 1 % and EUR denominated sales were up 11 % .",neutral,negative,0.972846
613,2401,"At 10.58 am , Outokumpu declined 2.74 pct to 24.87 eur , while the OMX Helsinki 25 was 0.55 pct higher at 2,825.14 and the OMX Helsinki added 0.64 pct to 9,386.89 .",neutral,negative,0.951658
342,4464,The board further said the company omitted to tender for a substantial part of the works and as such they had rightfully been found non-responsive by the evaluation team .,neutral,negative,0.942037
453,3186,"The company will also shut one paper machine in Finland and one in Austria , as well as two label paper machines in Finland for up to 10 months , Helsinki-based UPM said yesterday .",neutral,negative,0.934186
410,4197,Nokia is requesting that the companies stop making and selling the mobile phones and pay monetary damages and costs .,neutral,negative,0.922890
692,4575,"ASPOCOMP GROUP OYJ STOCK EXCHANGE RELEASE December 15 , 2006 at 4:50 PM According to the disclosure received today by Aspocomp Group Oyj , the share of Henrik Nyberg in Aspocomp Group Oyj 's share capital and votes has decreased below 5 percent .",neutral,negative,0.922802
314,3213,The duration of the lay-offs per employee will vary from one to six weeks .,neutral,negative,0.917564
552,4086,During the negotiations a reduction of 21 persons has taken place through natural redundancy or termination of fixed-term contracts .,neutral,negative,0.869382
86,3169,"The company has decided to stop the operations of Ruukki Construction Division in Latvia and Lithuania , and concentrate the production and logistics in Parnu , Estonia in 2009 .",neutral,negative,0.853950


# Observation
FinBERT reduced keyword-driven errors observed in the classical baseline. Remaining errors were primarily neutral sentences containing positive or negative financial events, suggesting annotation ambiguity rather than lack of domain understanding